In [ ]:
import os
import random
import time
import numpy as np
import torch
import cv2

from PIL import Image
from glob import glob



IMAGE_DIR = "/kaggle/input/datasets/jawadulkarim117/cityscapes/Cityscapes_Aug/Cityscapes_Aug/val/images"
CKPT_PATH = "/kaggle/working/outputs/best.pt"

SAVE_DIR = "/kaggle/working/pred"
os.makedirs(SAVE_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

IMAGE_SIZE  = (512, 1024)
NUM_CLASSES = 19

CITYSCAPES_COLORS = np.array([
    [128,64,128],[244,35,232],[70,70,70],[102,102,156],[190,153,153],
    [153,153,153],[250,170,30],[220,220,0],[107,142,35],[152,251,152],
    [70,130,180],[220,20,60],[255,0,0],[0,0,142],[0,0,70],
    [0,60,100],[0,80,100],[0,0,230],[119,11,32]
], dtype=np.uint8)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DualPathCRFNet().to(device)

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

state = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt

model.load_state_dict(state, strict=False)

model.eval()

print(f"Model loaded on: {device}")



def infer_single(image_path, model, device):

    img = Image.open(image_path).convert("RGB")

    ow, oh = img.size

    H, W = IMAGE_SIZE

    img_r = img.resize((W, H), Image.BILINEAR)

    x = torch.from_numpy(np.array(img_r)).permute(2,0,1).float() / 255.0

    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)

    x = ((x - mean) / std).unsqueeze(0).to(device)

    # Warmup
    if device.type == "cuda":
        with torch.no_grad():
            _ = model(x)
        torch.cuda.synchronize()

    # Timed inference
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()

    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(device.type=="cuda")):
            out = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.time() - t0

    pred = out["prob_f"].argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred = cv2.resize(pred, (ow, oh), interpolation=cv2.INTER_NEAREST)

    return np.array(img), pred, elapsed



all_images = glob(os.path.join(IMAGE_DIR, "*.png"))

selected_images = random.sample(all_images, 4)

print(f"Selected {len(selected_images)} random images")



for idx, image_path in enumerate(selected_images):

    print("="*60)
    print(f"Processing: {os.path.basename(image_path)}")

    orig_rgb, pred_mask, elapsed = infer_single(
        image_path,
        model,
        device
    )

    pred_colored = CITYSCAPES_COLORS[pred_mask]

    alpha = 0.55

    overlay = (
        alpha * orig_rgb.astype(np.float32) +
        (1 - alpha) * pred_colored.astype(np.float32)
    ).clip(0,255).astype(np.uint8)



    merged = np.concatenate(
        [orig_rgb, pred_colored, overlay],
        axis=1
    )


    base_name = os.path.splitext(os.path.basename(image_path))[0]

    save_path = os.path.join(
        SAVE_DIR,
        f"{base_name}_merged.png"
    )

    cv2.imwrite(
        save_path,
        cv2.cvtColor(merged, cv2.COLOR_RGB2BGR)
    )

    print(f"Inference Time : {elapsed*1000:.1f} ms")
    print(f"Saved: {save_path}")

print("="*60)
print(f"All merged predictions saved in: {SAVE_DIR}")